# Yoruba HealthQA -- Demo-Only Training (Google Colab, free GPU)

**Read this before running anything.**

This notebook trains a small proof-of-concept model so the Yoruba HealthQA app can show real generated answers during a defence/presentation. It is **not** the dissertation's final, validated model:

- The dataset used here (208 records) has been **post-edited by a Yoruba speaker** but has **NOT yet gone through clinical or linguistic validation sign-off**.
- 169 examples after safety filtering is a *very* small amount of training data. Even with a Yoruba-fluent base model, the medical *accuracy* of answers should not be trusted -- this run mainly fixes grammatical coherence, not clinical correctness.
- The base model (`Jacaranda/YorubaLlama`, a Llama-3-8B continually pretrained and instruction-tuned on real Yoruba text) is picked because it already has genuine Yoruba fluency -- unlike a generic multilingual model, which a real test run showed produces fluent-*looking* invented words instead of real Yoruba once undertrained. It is **not** the model the dissertation's own tokeniser-fertility analysis (`scripts/07_fertility.py`) would necessarily select for the final result.

**When presenting this, say so explicitly**: *"This is a proof-of-concept fine-tune on a small, not-yet-clinically-validated batch, using a Yoruba-capable base model, to demonstrate the pipeline works end to end. The dissertation's full model requires completing validation on the remaining data and training at full scale."*

## Before you start: one-time Hugging Face setup (~5 minutes)

`Jacaranda/YorubaLlama` is a **gated** model -- you need to request access once:

1. Create a free account at [huggingface.co/join](https://huggingface.co/join) if you don't have one.
2. Visit [huggingface.co/Jacaranda/YorubaLlama](https://huggingface.co/Jacaranda/YorubaLlama) while logged in and click **"Agree and access repository."** This is usually approved instantly; if it says it needs manual review, this path may not work in time for a same-day deadline.
3. Get an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) -> "New token" -> read access is enough. Keep it handy for the login cell below.

## How to use this notebook
1. In Colab: **Runtime -> Change runtime type -> T4 GPU** (free tier), then **Runtime -> Run all**.
2. When prompted, paste your Hugging Face token (login cell), then upload your three dataset files (`train.jsonl`, `val.jsonl`, `test.jsonl` from your local `data/final_demo/` folder).
3. Wait for training to finish. This is an 8B model (~16GB download) so expect longer than a small model -- likely 10-20 minutes total on a T4, mostly download time.
4. Download the zip file this notebook produces at the end.
5. On your own laptop, unzip it into your project's `models/` folder, then run the app with the environment variables shown at the bottom of this notebook.

In [ ]:
# Confirm a GPU is actually attached. If this errors, go to
# Runtime -> Change runtime type -> T4 GPU, then re-run.
!nvidia-smi

In [ ]:
# Get the pipeline CODE (not the data) from GitHub.
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
# Install only what's needed for training, on top of Colab's pre-installed
# torch+CUDA (deliberately NOT reinstalling torch from requirements.txt --
# that pinned version may not match Colab's CUDA build and could break GPU support).
#
# NOT pinning exact versions for transformers/peft/bitsandbytes/accelerate,
# on purpose: an earlier pinned version (bitsandbytes==0.43.3) failed on a
# real run with "Using `bitsandbytes` 4-bit quantization requires
# bitsandbytes>=0.46.1" -- Colab's base image moves forward over time and a
# pin chosen today can be stale by the time you run this. Letting pip resolve
# current mutually-compatible versions is more robust here than guessing a
# fixed set of numbers.
!pip install -q -U transformers peft bitsandbytes accelerate datasets sacrebleu sentencepiece pyyaml

In [ ]:
# Log in to Hugging Face -- required because Jacaranda/YorubaLlama is gated.
# Make sure you've already clicked "Agree and access repository" on
# https://huggingface.co/Jacaranda/YorubaLlama (see the setup steps above)
# before pasting your token here. Get a token at
# https://huggingface.co/settings/tokens (read access is enough).
from huggingface_hub import login
import getpass

login(token=getpass.getpass("Paste your Hugging Face access token: "))

In [ ]:
# Upload your three dataset files (from your laptop's data/final_demo/ folder).
import os
from google.colab import files

os.makedirs("data/final_demo", exist_ok=True)
print("Select train.jsonl, val.jsonl, and test.jsonl (you can select all three at once):")
uploaded = files.upload()
for fname in uploaded:
    os.replace(fname, f"data/final_demo/{fname}")
print("Uploaded:", os.listdir("data/final_demo"))

In [ ]:
# Pick the base model and write it into configs/train.yaml.
# Jacaranda/YorubaLlama: Llama-3-8B continually pretrained on 8.1GB of real
# Yoruba text and instruction-tuned on 66,280 Yoruba instruction-response
# pairs (verified via its Hugging Face model card). Chosen specifically
# because a real run with a generic multilingual model (Qwen2.5-1.5B) produced
# fluent-looking but INVENTED words, not real Yoruba, once undertrained --
# this base model already knows the language, so our small fine-tune only
# needs to adapt it to the health-QA domain/style, not teach it Yoruba from
# scratch. Gated -- see the setup steps and login cell above.
BASE_MODEL = "Jacaranda/YorubaLlama"

import yaml

with open("configs/train.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
train_cfg["base_model"] = BASE_MODEL
with open("configs/train.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False, allow_unicode=True)
print(f"configs/train.yaml base_model set to {BASE_MODEL}")

In [ ]:
# Train. This is the real QLoRA fine-tune -- expect a few minutes for this
# dataset size (169 training examples) on a T4.
!python scripts/09_train.py --single-run --data-dir data/final_demo \
    --train-config configs/train.yaml --model-config configs/model.yaml --models-dir models

In [ ]:
# Package the trained adapter for download.
import shutil

shutil.make_archive("single_run_adapter", "zip", "models/single_run")
files.download("single_run_adapter.zip")

## On your own laptop, after downloading

1. Unzip `single_run_adapter.zip` into your project folder. If your unzip tool creates an extra nested folder (check whether `adapter_config.json` ends up directly inside `models/single_run/` or one level deeper, e.g. `models/single_run/single_run_adapter/`), just point `YHQA_ADAPTER_PATH` below at whichever folder actually contains `adapter_config.json`.
2. Set these environment variables (or put them in your `.env` file) before starting the app:

```
YHQA_BASE_MODEL=Jacaranda/YorubaLlama
YHQA_ADAPTER_PATH=models/single_run
```

3. **Because the base model is gated, your laptop also needs to be logged in to Hugging Face once** before the app can download it -- run this once in a terminal (needs the `huggingface_hub` package, already in requirements.txt):
   ```
   huggingface-cli login
   ```
   and paste the same access token you used in Colab. (Alternatively, set an `HF_TOKEN` environment variable to the same token.)
4. Run the app as usual (`python app/app.py` or `docker compose up --build` with those variables set).
5. The status line in the app should now show the green "answer engine is working" message instead of the yellow placeholder one.

**A note on speed and size:** the first time you ask a question, the app downloads the full ~16GB base model from Hugging Face -- this only happens once, but make sure you have the disk space and a decent connection. Generating each answer on a CPU-only laptop will be slow (an 8B model is much heavier than the earlier 1.5B one -- possibly 1-3 minutes per answer). If that's too slow for a live demo, consider generating the answers to your example questions once beforehand and having them ready to paste/show, rather than generating live on stage.

**If training itself fails with an out-of-memory error** on the free T4 GPU (a real possibility with an 8B model), open `configs/train.yaml` and reduce `per_device_train_batch_size` under `optim` from 4 to 2 or 1 (increase `gradient_accumulation_steps` by the same factor to keep the effective batch size the same), then re-run the training cell.